In [4]:
# ==============================================================================
# CELL 1: SETUP & DATA CONFIGURATION
# ==============================================================================
# Install Ultralytics first
!pip install ultralytics -q

from google.colab import drive
import os
import yaml
from ultralytics import YOLO, RTDETR

# 1. Mount Google Drive
drive.mount('/content/drive')

# 2. Define Paths
# Adjust 'PE2' if your folder name is different in Drive
# Corrected base_path to match the actual Drive structure established in the main notebook
base_path = '/content/drive/MyDrive/Colab Notebooks/PE2/dataset'
train_path = 'train_augmented/images'
val_path = 'valid/images'
test_path = 'test/images'

# 3. Create a Colab-compatible YAML file
# This fixes the "D:/PE2" path issue automatically
data_config = {
    'path': base_path,
    'train': train_path,
    'val': val_path,
    'test': test_path,
    'nc': 1,
    'names': ['car']
}

yaml_path = os.path.join(base_path, 'data_augmented.yaml')

# Ensure the directory exists before writing the YAML file
os.makedirs(base_path, exist_ok=True)

with open(yaml_path, 'w') as f:
    yaml.dump(data_config, f, default_flow_style=False)

print("✅ Environment Ready!")
print(f"📄 Created Colab config at: {yaml_path}")
print("📦 Installing Ultralytics...")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
✅ Environment Ready!
📄 Created Colab config at: /content/drive/MyDrive/Colab Notebooks/PE2/dataset/data_augmented.yaml
📦 Installing Ultralytics...


In [8]:
# ==============================================================================
# CELL 2: TRAIN TRANSFORMER (RT-DETR)
# ==============================================================================

# We use the Large model (rtdetr-l) to maximize accuracy.
# It mimics the robust performance you saw in Roboflow.
model_transformer = RTDETR('rtdetr-l.pt')

print("🚀 STARTING TRANSFORMER TRAINING (RT-DETR)...")

# Train the model
results_transformer = model_transformer.train(
    data=yaml_path, # Use the correctly generated yaml_path from the previous cell
    epochs=50,                # Transformers learn fast, 50 is usually enough
    imgsz=640,
    batch=8,                  # Transformers need more VRAM, so we lower the batch size
    project='/content/drive/MyDrive/PE2/runs',  # Save directly to Drive
    name='parkxplore_transformer',              # Folder name
    optimizer='AdamW',        # AdamW is best for Transformers
    lr0=0.0001,               # Lower learning rate for stability
    exist_ok=True             # Overwrite if exists (careful!)
)

print("✅ Transformer Training Complete!")
print("💾 Model saved to: PE2/runs/parkxplore_transformer/weights/best.pt")

🚀 STARTING TRANSFORMER TRAINING (RT-DETR)...
Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=8, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/drive/MyDrive/Colab Notebooks/PE2/dataset/data_augmented.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=rtdetr-l.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=parkxplore_transformer, nbs=64, nms=False, 

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:157.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       1/50      6.63G     0.5255      1.085     0.2359         44        640: 100% ━━━━━━━━━━━━ 368/368 1.5it/s 4:12
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 1.4it/s 7.7s
                   all        171        880      0.157      0.682      0.204      0.137

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       2/50      7.02G     0.3419      1.066     0.1325         61        640: 0% ──────────── 0/368  0.9s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:157.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


       2/50      7.12G     0.3331      1.079     0.1329         44        640: 100% ━━━━━━━━━━━━ 368/368 1.6it/s 3:56
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 3.4it/s 3.2s
                   all        171        880      0.132      0.474      0.163      0.118

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
       3/50      7.12G      0.282      1.113     0.1067         65        640: 0% ──────────── 0/368  0.6s

...

      48/50      7.12G     0.1472     0.2358    0.06946         16        640: 100% ━━━━━━━━━━━━ 368/368 1.6it/s 3:48
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 3.4it/s 3.2s
                   all        171        880      0.986       0.99      0.995      0.832

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      49/50      7.12G     0.1633     0.2447     0.0759         43        640: 0% ──────────── 0/368  0.6s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:157.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      49/50      7.12G     0.1458     0.2339    0.06837         15        640: 100% ━━━━━━━━━━━━ 368/368 1.6it/s 3:47
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 3.0it/s 3.7s
                   all        171        880      0.991       0.99      0.995      0.832

      Epoch    GPU_mem  giou_loss   cls_loss    l1_loss  Instances       Size
      50/50      7.12G     0.1157     0.2534    0.06313         42        640: 0% ──────────── 0/368  0.7s

/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:841: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:157.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


      50/50      7.12G     0.1457     0.2331    0.06829         21        640: 100% ━━━━━━━━━━━━ 368/368 1.6it/s 3:48
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 3.4it/s 3.2s
                   all        171        880      0.988       0.99      0.995      0.833

50 epochs completed in 3.306 hours.
Optimizer stripped from /content/drive/MyDrive/PE2/runs/parkxplore_transformer/weights/last.pt, 66.2MB
Optimizer stripped from /content/drive/MyDrive/PE2/runs/parkxplore_transformer/weights/best.pt, 66.2MB

Validating /content/drive/MyDrive/PE2/runs/parkxplore_transformer/weights/best.pt...
Ultralytics 8.4.7 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
rt-detr-l summary: 310 layers, 31,985,795 parameters, 0 gradients, 103.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 2.0it/s 5.6s
                   all        171        880     